# 🏥 NetraAI (SIH26038) — Google Colab GPU Training Pipeline
### 5-Class Diabetic Retinopathy Severity Grading (ICDR 0–4)
**Hardware Target**: Google Colab Free NVIDIA T4 (16GB VRAM) or A100 GPU  
**Estimated Training Time**: ~12–15 minutes for 15 epochs.

## 1. Verify GPU Activation
Ensure you have selected **Runtime > Change runtime type > T4 GPU** in Google Colab.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Install Required Dependencies

In [ ]:
!pip install timm opencv-python-headless scikit-learn pandas pillow matplotlib -q

## 3. Load Dataset into Colab (Choose Option A or Option B)
**Option A (Fastest — 30 seconds)**: Download directly from Kaggle servers inside Colab.
**Option B**: Mount Google Drive if you already uploaded your zip file to Drive.

In [ ]:
# === OPTION A: DIRECT KAGGLE DOWNLOAD (Recommended - 30 seconds) ===
# If you have a Kaggle account (kaggle.json), upload it or paste your credentials:
# import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'
# !kaggle competitions download -c aptos2019-blindness-detection
# !unzip -q aptos2019-blindness-detection.zip -d /content/dataset

# === OPTION B: MOUNT GOOGLE DRIVE ===
from google.colab import drive
drive.mount('/content/drive')

# If you have aptos2019-blindness-detection.zip in your Google Drive (MyDrive):
# !unzip -q "/content/drive/MyDrive/aptos2019-blindness-detection.zip" -d /content/dataset

## 4. Core Preprocessing & Dataset Loader

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix

def crop_to_circle_mask(image, tol=7):
    if image.ndim == 2:
        mask = image > tol
        return image[np.ix_(mask.any(1), mask.any(0))], mask
    elif image.ndim == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        mask = gray > tol
        check_shape = image[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if check_shape > 0:
            img1 = image[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = image[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = image[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            return np.stack([img1, img2, img3], axis=-1), mask
    return image, np.ones(image.shape[:2], dtype=bool)

def apply_ben_graham(image, target_size=512):
    resized = cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_AREA)
    blurred = cv2.GaussianBlur(resized, (0, 0), target_size / 30.0)
    enhanced = cv2.addWeighted(resized, 4.0, blurred, -4.0, 128)
    mask = np.zeros((target_size, target_size), dtype=np.uint8)
    cv2.circle(mask, (target_size // 2, target_size // 2), int(target_size * 0.48), 255, -1)
    return cv2.bitwise_and(enhanced, enhanced, mask=mask)

class DRDataset(Dataset):
    def __init__(self, paths, labels=None, is_training=False):
        self.paths = paths
        self.labels = labels
        self.is_training = is_training
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1).astype(np.float32)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1).astype(np.float32)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        cropped, _ = crop_to_circle_mask(img)
        processed = apply_ben_graham(cropped, target_size=512)
        rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
        tensor = rgb.astype(np.float32) / 255.0

        if self.is_training:
            if np.random.rand() > 0.5:
                tensor = np.fliplr(tensor).copy()
            if np.random.rand() > 0.5:
                tensor = np.flipud(tensor).copy()

        tensor = np.transpose(tensor, (2, 0, 1))
        tensor = (tensor - self.mean) / self.std
        tensor = torch.tensor(tensor, dtype=torch.float32)

        if self.labels is not None:
            return tensor, torch.tensor(self.labels[idx], dtype=torch.long)
        return tensor, self.paths[idx]

## 5. Model Definition (EfficientNet-B3 with Transfer Learning)

In [ ]:
def get_model(num_classes=5):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 256),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes)
    )
    return model

## 6. Training Execution with Mixed Precision (AMP)

In [ ]:
# Adjust paths to your CSV and images folder
CSV_PATH = "/content/dataset/train.csv"
IMAGES_DIR = "/content/dataset/train_images"
EPOCHS = 15
BATCH_SIZE = 16
LR = 3e-4
OUTPUT_CHECKPOINT = "grading_efficientnet_b3.pt"

df = pd.read_csv(CSV_PATH)
paths = [os.path.join(IMAGES_DIR, f"{id_code}.png") for id_code in df['id_code']]
labels = df['diagnosis'].tolist()

# Stratified Train / Validation Split (80/20)
train_p, val_p, train_l, val_l = train_test_split(paths, labels, test_size=0.2, stratify=labels, random_state=42)

train_loader = DataLoader(DRDataset(train_p, train_l, is_training=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(DRDataset(val_p, val_l, is_training=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model().to(device)

# Class-balanced loss weights
counts = np.bincount(train_l, minlength=5)
weights = 1.0 / (counts + 1e-5)
weights = torch.tensor(weights / weights.sum() * 5.0, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

best_qwk = -1.0
print(f"Training on {device}: {len(train_p)} train samples, {len(val_p)} val samples")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)
    scheduler.step()

    # Validation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(targets.numpy())

    qwk = cohen_kappa_score(val_targets, val_preds, weights='quadratic')
    acc = np.mean(np.array(val_targets) == np.array(val_preds))
    
    # Referable DR metrics (Grade >= 2)
    b_true = (np.array(val_targets) >= 2).astype(int)
    b_pred = (np.array(val_preds) >= 2).astype(int)
    tn, fp, fn, tp = confusion_matrix(b_true, b_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {train_loss/len(train_p):.4f} | Acc: {acc*100:.1f}% | QWK: {qwk:.4f} | Ref Sens: {sens*100:.1f}% | Spec: {spec*100:.1f}%")
    
    if qwk > best_qwk:
        best_qwk = qwk
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_qwk': best_qwk,
            'backbone': 'efficientnet_b3'
        }, OUTPUT_CHECKPOINT)
        print(f"  >>> ⭐ Saved best model checkpoint: {OUTPUT_CHECKPOINT} (QWK: {best_qwk:.4f})")

print(f"\n Training Completed! Best QWK: {best_qwk:.4f}")

## 7. Download Trained Checkpoint for NetraAI Local/Render App
Download `grading_efficientnet_b3.pt` and place it in `ml/checkpoints/` in your local project!

In [ ]:
from google.colab import files
files.download('grading_efficientnet_b3.pt')